# Projeto Final (parte B)

## Experimentando as ferramentas de aprendizado de máquina do Spark

### Código básico para carregar os dados do INMET

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType #, DoubleType, IntegerType
from pyspark.sql import functions as F

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Meu Projeto")
    .config("spark.driver.memory", "4g")
    #.config("spark.executor.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 08:15:45 WARN Utils: Your hostname, lucas-dell, resolves to a loopback address: 127.0.1.1; using 192.168.100.143 instead (on interface wlp3s0)
26/05/21 08:15:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 08:15:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# verificar se as colunas são as mesmas
nomes_colunas = [
    "Data",
    "Hora UTC",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "RADIACAO GLOBAL (Kj/m²)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)"
]

def limpa_nome_coluna(nome:str):
    nome = nome.replace(' - ', '_')
    nome = nome.replace(', ', '_')
    nome = nome.replace('. ', '_')
    nome = nome.replace(' ', '_')
    nome = nome.replace('.', '_')
    nome = nome.replace(',', '_')
    return(nome)

nomes_colunas_limpo = [limpa_nome_coluna(nome) for nome in nomes_colunas]

In [4]:
for n in nomes_colunas_limpo:
    print(n)

Data
Hora_UTC
PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm)
PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB)
PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB)
PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB)
RADIACAO_GLOBAL_(Kj/m²)
TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C)
TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C)
TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C)
TEMPERATURA_MÍNIMA_NA_HORA_ANT_(AUT)_(°C)
TEMPERATURA_ORVALHO_MAX_NA_HORA_ANT_(AUT)_(°C)
TEMPERATURA_ORVALHO_MIN_NA_HORA_ANT_(AUT)_(°C)
UMIDADE_REL_MAX_NA_HORA_ANT_(AUT)_(%)
UMIDADE_REL_MIN_NA_HORA_ANT_(AUT)_(%)
UMIDADE_RELATIVA_DO_AR_HORARIA_(%)
VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr))
VENTO_RAJADA_MAXIMA_(m/s)
VENTO_VELOCIDADE_HORARIA_(m/s)


In [5]:
schema = [StructField(nome, StringType(), True) for nome in nomes_colunas_limpo]
schema = StructType(schema)

In [6]:
sdf = (
    spark.read
    .option('encoding', 'iso-8859-1') # 'latin1'
    .option('sep', ';')
    # .option('decimal', ',') # melhor fazer a leitura como string
    .option('header', False)
    .schema(schema)
    .csv('2024/') # pasta dos arquivos
)

In [21]:
def inmet_dados_formatados(sdf):
    # para remover as primeiras 8 linhas (metadados)
    # e também a linha com os nomes das colunas
    sdf = (
        sdf
        .filter(~sdf[sdf.columns[0]].contains(":"))
        .filter(~sdf[sdf.columns[0]].contains("Data"))
    )
    # convertendo os númericos para double
    for i in range(2,19):
        nome = sdf.columns[i]
        sdf = sdf.withColumn(
            nome, 
            F.regexp_replace(F.col(nome), ",", ".").cast("double")
        )
    # adicionando coluna temporária com nome do arquivo
    sdf = sdf.withColumn(
        "arquivo", 
        F.element_at(F.split(F.input_file_name(), "/"), -1) # nome.split('/')[-1]
    )
    # criando colunas de região, UF, código WMO e nome da estação
    # WMO = World Meteorological Organization (https://wmo.int/)
    sdf = (
        sdf
        .withColumn("regiao", F.element_at(F.split(F.col("arquivo"), "_"), 2))
        .withColumn("uf", F.element_at(F.split(F.col("arquivo"), "_"), 3))
        .withColumn("codigo_wmo", F.element_at(F.split(F.col("arquivo"), "_"), 4))
        .withColumn("estacao", F.element_at(F.split(F.col("arquivo"), "_"), 5))
    )

    sdf = sdf.dropna()
    sdf = sdf.drop('arquivo')
    return(sdf)

In [22]:
sdf_limpo = inmet_dados_formatados(sdf)

In [23]:
df_limpo = sdf_limpo.limit(5).toPandas()

In [24]:
sdf_limpo.columns

['Data',
 'Hora_UTC',
 'PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm)',
 'PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB)',
 'PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB)',
 'PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB)',
 'RADIACAO_GLOBAL_(Kj/m²)',
 'TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C)',
 'TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C)',
 'TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_MÍNIMA_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_ORVALHO_MAX_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_ORVALHO_MIN_NA_HORA_ANT_(AUT)_(°C)',
 'UMIDADE_REL_MAX_NA_HORA_ANT_(AUT)_(%)',
 'UMIDADE_REL_MIN_NA_HORA_ANT_(AUT)_(%)',
 'UMIDADE_RELATIVA_DO_AR_HORARIA_(%)',
 'VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr))',
 'VENTO_RAJADA_MAXIMA_(m/s)',
 'VENTO_VELOCIDADE_HORARIA_(m/s)',
 'regiao',
 'uf',
 'codigo_wmo',
 'estacao']

In [25]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 23 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Data                                                  5 non-null      object 
 1   Hora_UTC                                              5 non-null      object 
 2   PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm)                       5 non-null      float64
 3   PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB)  5 non-null      float64
 4   PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB)        5 non-null      float64
 5   PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB)        5 non-null      float64
 6   RADIACAO_GLOBAL_(Kj/m²)                               5 non-null      float64
 7   TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C)             5 non-null      float64
 8   TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C)                  5 non-null

### Esboço da rotina de PCA do Spark (via MLlib)

---

Referências:
- https://spark.apache.org/docs/latest/ml-guide.html
- https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.PCA.html
- https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.linalg.DenseVector.html

In [26]:
import numpy as np
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA

In [27]:
# a rotina de PCA do Spark precisa de uma coluna do tipo DenseVector
# https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html
agrupador = VectorAssembler(
    inputCols=[
 'PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm)',
 'PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB)',
 'PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB)',
 'PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB)',
 'RADIACAO_GLOBAL_(Kj/m²)',
 'TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C)',
 'TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C)',
 'TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_MÍNIMA_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_ORVALHO_MAX_NA_HORA_ANT_(AUT)_(°C)',
 'TEMPERATURA_ORVALHO_MIN_NA_HORA_ANT_(AUT)_(°C)',
 'UMIDADE_REL_MAX_NA_HORA_ANT_(AUT)_(%)',
 'UMIDADE_REL_MIN_NA_HORA_ANT_(AUT)_(%)',
 'UMIDADE_RELATIVA_DO_AR_HORARIA_(%)',
 'VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr))',
 'VENTO_RAJADA_MAXIMA_(m/s)',
 'VENTO_VELOCIDADE_HORARIA_(m/s)'], 
    outputCol="vetor_agrupado"
)
df_agrupado = agrupador.transform(sdf_limpo)

print(df_agrupado.dtypes)
df_agrupado.toPandas()

[('Data', 'string'), ('Hora_UTC', 'string'), ('PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm)', 'double'), ('PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB)', 'double'), ('PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB)', 'double'), ('PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB)', 'double'), ('RADIACAO_GLOBAL_(Kj/m²)', 'double'), ('TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C)', 'double'), ('TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C)', 'double'), ('TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C)', 'double'), ('TEMPERATURA_MÍNIMA_NA_HORA_ANT_(AUT)_(°C)', 'double'), ('TEMPERATURA_ORVALHO_MAX_NA_HORA_ANT_(AUT)_(°C)', 'double'), ('TEMPERATURA_ORVALHO_MIN_NA_HORA_ANT_(AUT)_(°C)', 'double'), ('UMIDADE_REL_MAX_NA_HORA_ANT_(AUT)_(%)', 'double'), ('UMIDADE_REL_MIN_NA_HORA_ANT_(AUT)_(%)', 'double'), ('UMIDADE_RELATIVA_DO_AR_HORARIA_(%)', 'double'), ('VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr))', 'double'), ('VENTO_RAJADA_MAXIMA_(m/s)', 'double'), ('VENTO_VELOCIDADE_HORARIA_(m/s)', 'double'), ('regiao', 'string'), ('uf', 'string

,Data,Hora_UTC,PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm),PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB),PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB),PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB),RADIACAO_GLOBAL_(Kj/m²),TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C),TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C),TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C),...,UMIDADE_REL_MIN_NA_HORA_ANT_(AUT)_(%),UMIDADE_RELATIVA_DO_AR_HORARIA_(%),VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr)),VENTO_RAJADA_MAXIMA_(m/s),VENTO_VELOCIDADE_HORARIA_(m/s),regiao,uf,codigo_wmo,estacao,vetor_agrupado
0,2024/01/01,0000 UTC,0.0,1009.8,1009.8,1009.2,0.0,27.4,24.1,28.2,...,80.0,82.0,193.0,4.7,1.3,N,PA,A201,BELEM,"[0.0, 1009.8, 1009.8, 1009.2, 0.0, 27.4, 24.1,..."
1,2024/01/01,0100 UTC,0.0,1010.8,1010.8,1009.8,0.0,27.2,24.2,27.4,...,82.0,84.0,242.0,4.2,0.4,N,PA,A201,BELEM,"[0.0, 1010.8, 1010.8, 1009.8, 0.0, 27.2, 24.2,..."
2,2024/01/01,0200 UTC,0.0,1010.7,1010.8,1010.3,0.0,26.4,24.7,27.2,...,84.0,90.0,242.0,1.6,0.0,N,PA,A201,BELEM,"[0.0, 1010.7, 1010.8, 1010.3, 0.0, 26.4, 24.7,..."
3,2024/01/01,0300 UTC,0.0,1010.2,1010.8,1010.2,0.0,26.9,24.8,26.9,...,88.0,88.0,248.0,1.8,0.3,N,PA,A201,BELEM,"[0.0, 1010.2, 1010.8, 1010.2, 0.0, 26.9, 24.8,..."
4,2024/01/01,0400 UTC,0.0,1009.7,1010.2,1009.7,0.0,26.9,24.8,27.1,...,88.0,89.0,236.0,3.0,0.0,N,PA,A201,BELEM,"[0.0, 1009.7, 1010.2, 1009.7, 0.0, 26.9, 24.8,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893908,2024/01/03,1400 UTC,0.0,978.2,978.7,978.2,1971.3,30.9,21.1,30.9,...,56.0,56.0,95.0,2.3,0.0,NE,BA,A436,QUEIMADAS,"[0.0, 978.2, 978.7, 978.2, 1971.3, 30.9, 21.1,..."
1893909,2024/01/03,1500 UTC,0.0,977.1,978.2,977.1,3024.4,31.3,19.7,32.7,...,47.0,50.0,56.0,5.4,0.4,NE,BA,A436,QUEIMADAS,"[0.0, 977.1, 978.2, 977.1, 3024.4, 31.3, 19.7,..."
1893910,2024/01/03,1600 UTC,0.0,976.4,977.2,976.4,2153.8,31.8,19.9,32.0,...,48.0,50.0,63.0,8.8,4.1,NE,BA,A436,QUEIMADAS,"[0.0, 976.4, 977.2, 976.4, 2153.8, 31.8, 19.9,..."
1893911,2024/01/03,1700 UTC,0.0,975.0,976.4,975.0,2195.3,30.4,21.0,34.4,...,42.0,57.0,269.0,7.4,3.9,NE,BA,A436,QUEIMADAS,"[0.0, 975.0, 976.4, 975.0, 2195.3, 30.4, 21.0,..."


In [28]:
# rotina de normalização
# subtrai a média (centraliza em zero)
# dividide pelo desvio padrão (normaliza o desvio padrão para 1)
config_normalizacao = StandardScaler(
    inputCol="vetor_agrupado", 
    outputCol="vetor_normalizado", 
    withMean=True,
    withStd=True
)
modelo_normalizacao = config_normalizacao.fit(df_agrupado)
df_normalizado = modelo_normalizacao.transform(df_agrupado)
df_normalizado.toPandas()

,Data,Hora_UTC,PRECIPITAÇÃO_TOTAL_HORÁRIO_(mm),PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO_HORARIA_(mB),PRESSÃO_ATMOSFERICA_MAX_NA_HORA_ANT_(AUT)_(mB),PRESSÃO_ATMOSFERICA_MIN_NA_HORA_ANT_(AUT)_(mB),RADIACAO_GLOBAL_(Kj/m²),TEMPERATURA_DO_AR_BULBO_SECO_HORARIA_(°C),TEMPERATURA_DO_PONTO_DE_ORVALHO_(°C),TEMPERATURA_MÁXIMA_NA_HORA_ANT_(AUT)_(°C),...,UMIDADE_RELATIVA_DO_AR_HORARIA_(%),VENTO_DIREÇÃO_HORARIA_(gr)_(°_(gr)),VENTO_RAJADA_MAXIMA_(m/s),VENTO_VELOCIDADE_HORARIA_(m/s),regiao,uf,codigo_wmo,estacao,vetor_agrupado,vetor_normalizado
0,2024/01/01,0000 UTC,0.0,1009.8,1009.8,1009.2,0.0,27.4,24.1,28.2,...,82.0,193.0,4.7,1.3,N,PA,A201,BELEM,"[0.0, 1009.8, 1009.8, 1009.2, 0.0, 27.4, 24.1,...","[-0.12344453435996403, 1.1749068588158111, 1.1..."
1,2024/01/01,0100 UTC,0.0,1010.8,1010.8,1009.8,0.0,27.2,24.2,27.4,...,84.0,242.0,4.2,0.4,N,PA,A201,BELEM,"[0.0, 1010.8, 1010.8, 1009.8, 0.0, 27.2, 24.2,...","[-0.12344453435996403, 1.2009401976950127, 1.1..."
2,2024/01/01,0200 UTC,0.0,1010.7,1010.8,1010.3,0.0,26.4,24.7,27.2,...,90.0,242.0,1.6,0.0,N,PA,A201,BELEM,"[0.0, 1010.7, 1010.8, 1010.3, 0.0, 26.4, 24.7,...","[-0.12344453435996403, 1.198336863807095, 1.19..."
3,2024/01/01,0300 UTC,0.0,1010.2,1010.8,1010.2,0.0,26.9,24.8,26.9,...,88.0,248.0,1.8,0.3,N,PA,A201,BELEM,"[0.0, 1010.2, 1010.8, 1010.2, 0.0, 26.9, 24.8,...","[-0.12344453435996403, 1.185320194367494, 1.19..."
4,2024/01/01,0400 UTC,0.0,1009.7,1010.2,1009.7,0.0,26.9,24.8,27.1,...,89.0,236.0,3.0,0.0,N,PA,A201,BELEM,"[0.0, 1009.7, 1010.2, 1009.7, 0.0, 26.9, 24.8,...","[-0.12344453435996403, 1.1723035249278932, 1.1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893908,2024/01/03,1400 UTC,0.0,978.2,978.7,978.2,1971.3,30.9,21.1,30.9,...,56.0,95.0,2.3,0.0,NE,BA,A436,QUEIMADAS,"[0.0, 978.2, 978.7, 978.2, 1971.3, 30.9, 21.1,...","[-0.12344453435996403, 0.35225335023303905, 0...."
1893909,2024/01/03,1500 UTC,0.0,977.1,978.2,977.1,3024.4,31.3,19.7,32.7,...,50.0,56.0,5.4,0.4,NE,BA,A436,QUEIMADAS,"[0.0, 977.1, 978.2, 977.1, 3024.4, 31.3, 19.7,...","[-0.12344453435996403, 0.32361667746591655, 0...."
1893910,2024/01/03,1600 UTC,0.0,976.4,977.2,976.4,2153.8,31.8,19.9,32.0,...,50.0,63.0,8.8,4.1,NE,BA,A436,QUEIMADAS,"[0.0, 976.4, 977.2, 976.4, 2153.8, 31.8, 19.9,...","[-0.12344453435996403, 0.30539334025047415, 0...."
1893911,2024/01/03,1700 UTC,0.0,975.0,976.4,975.0,2195.3,30.4,21.0,34.4,...,57.0,269.0,7.4,3.9,NE,BA,A436,QUEIMADAS,"[0.0, 975.0, 976.4, 975.0, 2195.3, 30.4, 21.0,...","[-0.12344453435996403, 0.2689466658195923, 0.2..."


Rodamos inicialmente o PCA para k=17 e observamos a variância acumulada para escolher o valor ótimo de k. Obtivemos os seguintes resultados:

| k | Variância Explicada | Variância Acumulada |
|---|---|---|
| 1 | 32.93% | 32.93% |
| 2 | 29.01% | 61.94% |
| 3 | 11.42% | 73.36% |
| 4 | 10.00% | 83.36% |
| 5 | 5.90% | 89.26% |
| 6 | 5.58% | 94.84% |
| 7 | 3.44% | 98.28% |
| 8 | 0.88% | 99.16% |
| 9 | 0.40% | 99.56% |
| 10 | 0.20% | 99.76% |
| 11 | 0.13% | 99.89% |
| 12 | 0.04% | 99.93% |
| 13 | 0.04% | 99.97% |
| 14 | 0.02% | 99.99% |
| 15 | 0.01% | 100.00% |
| 16 | 0.00% | 100.00% |
| 17 | 0.00% | 100.00% |

In [33]:
# PCA é uma decomposição matricial da forma T = X.W
pca = PCA(
    k=4, 
    inputCol="vetor_normalizado", 
    outputCol="pontuacoes"
)
modelo_pca = pca.fit(df_normalizado)
df_pca = modelo_pca.transform(df_normalizado)

In [34]:
# matrizes X e T
df_pca.select("vetor_normalizado", "pontuacoes").toPandas()

,vetor_normalizado,pontuacoes
0,"[-0.12344453435996403, 1.1749068588158111, 1.1...","[-2.198350518707897, -2.6577447218654386, 0.28..."
1,"[-0.12344453435996403, 1.2009401976950127, 1.1...","[-2.385380916149189, -2.5494923326089975, 0.33..."
2,"[-0.12344453435996403, 1.198336863807095, 1.19...","[-3.0040341681493774, -2.5376715465153774, 0.2..."
3,"[-0.12344453435996403, 1.185320194367494, 1.19...","[-3.028538646757967, -2.55782897713181, 0.2161..."
4,"[-0.12344453435996403, 1.1723035249278932, 1.1...","[-2.916773310311366, -2.596347192813085, 0.182..."
...,...,...
1893908,"[-0.12344453435996403, 0.35225335023303905, 0....","[0.44434322476370147, -1.994421502186758, -0.5..."
1893909,"[-0.12344453435996403, 0.32361667746591655, 0....","[1.5816264795080814, -2.158105944991387, -0.58..."
1893910,"[-0.12344453435996403, 0.30539334025047415, 0....","[1.9413413421134238, -1.866677533292382, -0.31..."
1893911,"[-0.12344453435996403, 0.2689466658195923, 0.2...","[1.7202891111487382, -1.9296224375737212, -0.6..."


In [35]:
# matriz W
modelo_pca.pc.toArray()

array([[-0.06026115,  0.00919734, -0.03587494,  0.18515726],
       [-0.08263991, -0.33436424,  0.45730118,  0.01655804],
       [-0.08099055, -0.33519871,  0.45635223,  0.01746983],
       [-0.08131174, -0.33494312,  0.45678771,  0.01656324],
       [ 0.25183647, -0.10808175, -0.15247913,  0.11526594],
       [ 0.25801375, -0.33112907, -0.17231407, -0.07071418],
       [-0.23236581, -0.32552763, -0.28224912,  0.07172748],
       [ 0.26316132, -0.32386545, -0.17312949, -0.06579898],
       [ 0.24748781, -0.33408732, -0.16831161, -0.06733191],
       [-0.21194424, -0.3353855 , -0.29594263,  0.0764314 ],
       [-0.24019935, -0.32094236, -0.27241312,  0.08090771],
       [-0.40993962,  0.02209022, -0.07597366,  0.10396464],
       [-0.40973616,  0.02837237, -0.04643517,  0.09562628],
       [-0.41291696,  0.02525724, -0.06083145,  0.10098761],
       [-0.02618342,  0.03166368, -0.07725165, -0.12776823],
       [ 0.18191712,  0.01395328,  0.01230713,  0.64975516],
       [ 0.13801224,  0.